# slv_cuarentena
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · cuarentena |
| Propósito | Tabla única donde terminan las filas que no cumplen una regla de descarte, con la entidad, la clave, las reglas incumplidas y la fila original. Un flujo por entidad lee su vista limpio_<entidad> (definida en slv_<entidad>.py) y se queda con lo que falló. |
| Por qué | nada se descarta en silencio. Lo rechazado queda auditable y se puede corregir y reprocesar. |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.1 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt

from comun import filas_en_cuarentena

## 2. Entidades que envían a cuarentena
Para cada entidad: su vista de limpieza (definida en `slv_<entidad>`) y su clave.

In [ ]:
# entidad → (vista de limpieza, clave)
ENTIDADES = {
    "clientes": ("limpio_clientes", "id_cliente"),
    "productos": ("limpio_productos", "id_producto"),
    "pedidos": ("limpio_pedidos", "id_pedido"),
    "items_pedido": ("limpio_items_pedido", "id_item"),
    "pagos": ("limpio_pagos", "id_pago"),
    "tickets": ("limpio_tickets", "id_ticket"),
}

## 3. Tabla destino
Se crea la tabla de streaming que reciben los flujos de esta sección.

In [ ]:
dlt.create_streaming_table(
    name="cuarentena",
    comment="Filas que no cumplieron una regla de calidad: tabla, clave, reglas incumplidas y fila original.")

## 4. Flujos hacia la cuarentena
Un flujo por entidad: lee su vista de limpieza y se queda con las filas que no cumplen una regla de descarte, con la lista de reglas incumplidas.

In [ ]:
def flujo_cuarentena(entidad: str, vista: str, clave: str):
    """Registra el flujo de una entidad hacia silver.cuarentena."""
    @dlt.append_flow(target="cuarentena", name=f"cuarentena_{entidad}")
    def _flujo():
        return filas_en_cuarentena(dlt.read_stream(vista), entidad, clave)

## 5. Registro de los flujos
Se registra un flujo por cada entidad de la lista.

In [ ]:
for entidad, (vista, clave) in ENTIDADES.items():
    flujo_cuarentena(entidad, vista, clave)